{% include nav/toolkits/bathroom/menu.html %}

<head>
    <link href="https://fonts.googleapis.com/icon?family=Material+Icons" rel="stylesheet">
</head>

<style>
    .container {
        position: relative;
        top: 10;
        margin-top: 10 !important;
    }
    #additionalInput {
        display: none; /* Initially hidden */
    }
    body {
        color: black; 
    }
    .header-container {
        text-align: center;
        margin-top: 20px;
    }
    .header-container h1 {
        font-size: 28px;
        color: white;
    }
</style>

<script type="module">
    import { javaURI, fetchOptions } from '{{ site.baseurl }}/assets/js/api/config.js';

    document.addEventListener("DOMContentLoaded", function () {
        document.getElementById("requestPassBtn").addEventListener("click", requestHallPass);
        document.getElementById("checkinBtn").addEventListener("click", checkinHallPass);
    });

    // Retrieve stored teacher info from localStorage
    const storedTeacherFirstName = localStorage.getItem("teacherFirstName");
    const storedTeacherLastName = localStorage.getItem("teacherLastName");

    async function fetchTeacherData() {
        try {
            let response = await fetch(`${javaURI}/api/hallpass/getTeacher?fname=${storedTeacherFirstName}&lname=${storedTeacherLastName}`, fetchOptions);
            if (!response.ok) throw new Error("Network response was not ok");

            const contentType = response.headers.get("content-type");
            let data = contentType && contentType.includes("application/json") ? await response.json() : await response.text();

            if (!data || (typeof data === "object" && Object.keys(data).length === 0)) {
                document.getElementById("error-message").textContent = "Teacher not found. Enter your teacher's name below.";
                document.getElementById("error-container").style.display = "block";
                document.getElementById("teacher-form").style.display = "block";
            } else {
                await fetchActivePass();
            }
        } catch (error) {
            console.error("Error fetching teacher data:", error);
            document.getElementById("error-message").textContent = "An error occurred while fetching data. Please try again later.";
            document.getElementById("error-container").style.display = "block";
        }
    }

    async function fetchActivePass() {
        try {
            let response = await fetch(`${javaURI}/api/hallpass/getactivepass?email=toby`, fetchOptions);
            if (!response.ok) throw new Error("Network response was not ok");

            const contentType = response.headers.get("content-type");
            let data = contentType && contentType.includes("application/json") ? await response.json() : await response.text();

            if (!data || (typeof data === "object" && Object.keys(data).length === 0)) {
                document.getElementById("pass-form").style.display = "block";
            } else {
                document.getElementById("active-pass-container").style.display = "block";
                document.getElementById("active-teacher").textContent = `${storedTeacherFirstName} ${storedTeacherLastName}`;
                document.getElementById("active-activity").textContent = data.activity;
                document.getElementById("active-period").textContent = data.period;
            }
        } catch (error) {
            console.error("Error fetching active pass:", error);
            document.getElementById("error-message").textContent = "An error occurred while fetching data. Please try again later.";
            document.getElementById("error-container").style.display = "block";
        }
    }

    function saveTeacher() {
        const teacherFirstName = document.getElementById("teacherFirstName").value;
        const teacherLastName = document.getElementById("teacherLastName").value;

        if (teacherFirstName && teacherLastName) {
            localStorage.setItem("teacherFirstName", teacherFirstName);
            localStorage.setItem("teacherLastName", teacherLastName);
            alert("Teacher saved! Refreshing page...");
            location.reload();
        } else {
            alert("Please enter both the first and last name of your teacher.");
        }
    }

    async function requestHallPass() {
        const period = document.getElementById("period").value;
        const activity = document.getElementById("activity").value;
        const resultDiv = document.getElementById("confirm-message");

        resultDiv.innerHTML = "";

        const requestBody = {
            userName: "toby",
            teacherFirstName: storedTeacherFirstName,
            teacherLastName: storedTeacherLastName,
            period: period,
            activity: activity
        };

        try {
            const response = await fetch(`${javaURI}/api/hallpass/request`, {
                method: "POST",
                headers: { "Content-Type": "application/json" },
                body: JSON.stringify(requestBody)
            });

            if (!response.ok) throw new Error(`HTTP error! Status: ${response.status}`);

            document.getElementById("pass-form").style.display = "none";
            document.getElementById("active-pass-container").style.display = "none";
            document.getElementById("confirmation-container").style.display = "block";
            resultDiv.innerHTML = `<span style="color: green;font-size: 20px;">Success! Your hall pass has been processed.</span>`;
        } catch (error) {
            console.error("Error requesting hall pass:", error);
        }
    }
    fetchTeacherData();
</script>

<body>
    <div class="header-container">
        <h1 style="color:#0947ab;">Welcome to Hall Pass Request</h1>
    </div>
    <div class="container bg-primary text-black">
        <div id="confirmation-container" class="bg-light rounded" style="display: none;">
            <p id="confirm-message"></p>
        </div>
        <div id="error-container" class="alert alert-danger" role="alert" style="display: none;">
            <p id="error-message"></p>
        </div>
        <!-- Teacher Entry Form -->
        <div id="teacher-form" class="bg-light rounded" style="display: none;">
            <h2>Enter Your Teacher’s Name</h2>
            <div class="form-group">
                <label for="teacherFirstName">Teacher First Name:</label>
                <input type="text" id="teacherFirstName" name="teacherFirstName" class="form-control" required>
            </div>
            <div class="form-group">
                <label for="teacherLastName">Teacher Last Name:</label>
                <input type="text" id="teacherLastName" name="teacherLastName" class="form-control" required>
            </div>
            <button type="button" class="btn btn-primary mt-2" onclick="saveTeacher()">Save Teacher</button>
        </div>
        <div id="pass-form" class="bg-light rounded" style="display: none;">
            <h2>Request Hall Pass</h2>
            <label for="period">Period:</label>
            <select id="period" class="form-control">
                <option value="1">1</option>
                <option value="2">2</option>
                <option value="3">3</option>
                <option value="4">4</option>
                <option value="5">5</option>
            </select>
            <label for="activity">Activity:</label>
            <select id="activity" class="form-control">
                <option value="bathroom">Bathroom</option>
                <option value="library">Library</option>
                <option value="other">Other</option>
            </select>
            <button id="requestPassBtn" class="btn btn-primary mt-3">Request Hall Pass</button>
        </div>
    </div>
</body>

 